# VAE and conditional VAE representations

> **Notebook role:** production-style construction and interpretation of learned nuclear image embeddings.

## 1. Standardize nuclear crops

Crop size, channel count and intensity normalization are part of the model
contract. The input archive represents the caller-selected training split;
object identifiers remain attached to every crop and latent row.

In [ ]:
from pathlib import Path
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset

crop_archive = np.load(Path("data/nuclear_crops.npz"), allow_pickle=False)
crops = crop_archive["images"]
object_ids = crop_archive["object_ids"].astype(str)
crop_tensor = torch.as_tensor(crops, dtype=torch.float32).reshape(-1, 1, 64, 64)
crop_tensor = crop_tensor / crop_tensor.amax(dim=(1, 2, 3), keepdim=True).clamp_min(1e-8)

class NuclearCropDataset(Dataset):
    def __len__(self):
        return len(crop_tensor)

    def __getitem__(self, index):
        return {'image': crop_tensor[index], 'object_id': object_ids[index]}

dataset = NuclearCropDataset()
training_loader = DataLoader(dataset, batch_size=64, shuffle=True)
feature_loader = DataLoader(dataset, batch_size=64, shuffle=False)
crop_tensor.shape

## 2. Fit and reconstruct

Training uses summed reconstruction error and the configured KL weight. The
conditional model can add a classification objective while retaining the same
image representation path. Dataset splitting remains external to this step.

In [ ]:
from nuclear_vae_embeddings import fit_vae, reconstruct_images
from nuclear_vae_embeddings.models import CVAE, VAE

vae = VAE(nc=1, latent_variable_size=128, imsize=64)
cvae = CVAE(nc=1, latent_variable_size=128, imsize=64)
optimizer = torch.optim.Adam(vae.parameters(), lr=1e-4)
history = fit_vae(vae, training_loader, optimizer, epochs=20)
reconstruction = reconstruct_images(vae, crop_tensor[:8])
history.tail(), reconstruction.shape

## 3. Join learned and interpretable measurements

Latent dimensions complement morphology and texture. Keep object identifiers in
the embedding table so representations can be joined without changing row
identity.

In [ ]:
import pandas as pd
from nuclear_vae_embeddings import latent_feature_table

latent_table = latent_feature_table(
    vae, feature_loader, id_key="object_id", feature_prefix="latent_",
)
nuclear_features = pd.read_csv(Path("outputs/features/nuclei.csv"))
combined = nuclear_features.merge(latent_table, on="object_id", validate="one_to_one")
combined.head()

## 4. Save the joined representation table

The joined table is the explicit hand-off to feature filtering, statistics and
figure generation.

In [ ]:
output_path = Path("outputs/representations.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
combined.to_csv(output_path, index=False)

## Representative result

Inspect the learned feature space together with per-crop reconstruction error.
These two views separate representation structure from reconstruction quality.

![CVAE latent features and reconstruction error](../assets/results/vae-latent-diagnostics.png)